# **---Tinder Project (EDA) ---**

**Can we identify what drives people to select someone for a second date? That's the main question we have to solve for Tinder company by doing an EDA (Explanatory Data Analysis).**

#### **I. DATA OVERVIEW**

**1. Imports**

In [ ]:
import pandas as pd
import plotly.io as pio
import plotly.express as px
import plotly.graph_objects as go

**2. Overview**

In [ ]:
df=pd.read_csv("DATA/Speed_dating_data.csv", encoding="ISO-8859-1")
df.head()

In [ ]:
pd.options.display.max_columns = None # In order to avoid the columns to be truncated when displaying a table

In [ ]:
df.describe(include="all")

In [ ]:
df.info

In [ ]:
df.shape

In [ ]:
list_columns = df.columns.tolist()
list_columns

In [ ]:
fig_gender_proportion=px.pie(df,names="gender",values="iid",title="Proportion of men and women",width=400,height=400)
fig_gender_proportion.show()

In [ ]:
fig_age_distribution=px.histogram(df,x="age",y="iid",title="Age distribution of the participants",width=700,height=400,color_discrete_sequence = px.colors.qualitative.Prism)
fig_age_distribution.show()

In [ ]:
fig_race=px.histogram(df,x="race",y="iid",title="Origine of the participants distribution",width=700,height=400,color_discrete_sequence = px.colors.qualitative.Prism)
fig_race.show()

# Legend : 
	# Black/African American=1
	# European/Caucasian-American=2
	# Latino/Hispanic American=3
	# Asian/Pacific Islander/Asian-American=4
	# Native American=5
	# Other=6


In [ ]:
fig_income=px.histogram(df,x="iid",y="income",title="Income distribution of the participants",width=700,height=400,color_discrete_sequence = px.colors.qualitative.Prism)
fig_income.show()

In [ ]:
fig_goal=px.histogram(df,x="goal",y="iid",title="Goal of the participants",width=700,height=400,color_discrete_sequence = px.colors.sequential.Rainbow)
fig_goal.show()
# Seemed like a fun night out=1
# To meet new people=2
# To get a date=3
# Looking for a serious relationship=4
# To say I did it=5
# Other=6

In [ ]:
# To quickly find the color scales : px.colors.sequential.swatches()

In [ ]:
fig_dating_freq=px.histogram(df,x="date",y="iid",title="Dating frequency of the participants",width=700,height=400,color_discrete_sequence = px.colors.sequential.Rainbow)
fig_dating_freq.show()
	# Several times a week=1
	# Twice a week=2
	# Once a week=3
	# Twice a month=4
	# Once a month=5
	# Several times a year=6
	# Almost never=7


In [ ]:
fig_go_out_freq=px.histogram(df,x="go_out",y="iid",title="Going out frequency of the participants",width=700,height=400,color_discrete_sequence = px.colors.sequential.Rainbow)
fig_go_out_freq.show()
# Several times a week=1
# 	Twice a week=2
# 	Once a week=3
# 	Twice a month=4
# 	Once a month=5
# 	Several times a year=6
# 	Almost never=7


##2. Minimal cleaning & preparation Objective here: **prepare a clean but simple dataset** for the EDA, without getting into a big data cleaning project. Main steps: 1. Look at the **% missing values** on key variables. 2. Create some useful variables for the project (e.g.: ºCgender_labelʼ, ˅same_race’). 3. Select a subset of important columns for analysis.


In [ ]:
# 2.1. Selection of a few key columns

# We prepare lists of candidate columns and keep only those that actually exist in the dataframe.
target_cols = [c for c in ["match", "dec"] if c in df.columns]

rating_cols = [c for c in ["attr", "sinc", "intel", "fun", "amb", "shar"] if c in df.columns]

# Stated importance of criteria for a partner (according to the data key)
importance_cols = [c for c in ["attr1_1", "sinc1_1", "intel1_1", "fun1_1", "amb1_1", "shar1_1"] if c in df.columns]

# Self-evaluation (self-rating)
self_cols = [c for c in ["attr1_2", "sinc1_2", "intel1_2", "fun1_2", "amb1_2", "shar1_2"] if c in df.columns]

# Ratings received from others (if available)
others_rating_cols = [c for c in ["attr_o", "sinc_o", "intel_o", "fun_o", "amb_o", "shar_o"] if c in df.columns]

# Demographics
demo_cols = [c for c in ["iid", "gender", "age", "race", "field", "career_c"] if c in df.columns]

# Date position during the event
position_cols = [c for c in ["position", "order"] if c in df.columns]

key_cols = list(dict.fromkeys(target_cols + rating_cols + importance_cols +
                              self_cols + others_rating_cols + demo_cols + position_cols))

print("Key columns selected for the analysis:")
print(key_cols)

In [ ]:
# 2.2. Percentage of missing values for important columns

missing_pct = (
    df[key_cols]
    .isna()
    .mean()
    .sort_values(ascending=False) * 100
).round(1)

missing_df = missing_pct.to_frame(name="% NaN")
display(missing_df.head(20))

**Interpretation:**

- This table shows the **percentage of missing values** for each important column.
- If a key variable has too many NaN values (for example > 40–50%), its interpretation should be approached with caution.
- For a training EDA, it is acceptable to **remove some incomplete rows** rather than perform advanced imputations.

In [ ]:
# 2.3. Creating a working copy and first simple cleaning steps

df_clean = df.copy()

# Main target variable: use 'match' if available (mutual yes/yes),
# otherwise fall back on 'dec' (individual decision).
target_var = "match" if "match" in df_clean.columns else "dec"

# Remove rows where the target, gender, or age is missing
cols_minimals = [c for c in [target_var, "gender", "age"] if c in df_clean.columns]
df_clean = df_clean.dropna(subset=cols_minimals)

print("Shape after removing rows without target/gender/age:", df_clean.shape)

# 2.4. Add a readable 'gender_label' variable (Male / Female)
if "gender" in df_clean.columns:
    # According to the data key, gender = 0/1 (often 0 = female, 1 = male)
    df_clean["gender_label"] = df_clean["gender"].map({0: "Female", 1: "Male"})

In [ ]:
# 2.5. Creation of a variable 'same_race'

if "samerace" in df_clean.columns:
    # The dataset already provides a binary indicator
    df_clean["same_race"] = df_clean["samerace"]
elif "race" in df_clean.columns and "race_o" in df_clean.columns:
    # It is built from race and race_o
    df_clean["same_race"] = (df_clean["race"] == df_clean["race_o"]).astype(int)
else:
    df_clean["same_race"] = np.nan  # in case no information is available

# 2.6. If we want a sub-dataframe focused on the main variables :
df_eda_cols = list(dict.fromkeys(
    [target_var] + rating_cols + importance_cols + self_cols +
    others_rating_cols + demo_cols + ["gender_label", "same_race"] + position_cols
))
df_eda = df_clean[df_eda_cols]

print("Columns in df_eda:", df_eda.columns.tolist())
print("Shape de df_eda :", df_eda.shape)

**Interpretation :**

- We created `df_clean` en retirant les lignes sans cible, âge ou genre, pour éviter des biais trop forts.
- We add une variable plus lisible `gender_label` (Homme / Femme) pour faciliter les graphiques.
- La variable `same_race` indique si les 2 personnes sont de même origine (déjà fournie ou reconstruite).
- `df_eda` contains only the variables useful for the rest of the analysis.

## 3. General descriptives (participant profile)

Objective: understand **who participates** and **what is the overall match rate**. We'll look: 
- distribution by sex; 
- the age distribution; 
- distribution by race/origin (numerical, but commented); 
- the overall rate of “match” (second mutual date).

In [ ]:
# 3.1. Distribution by gender

plt.figure(figsize=(5, 4))
sns.countplot(data=df_eda, x="gender_label")
plt.title("Distribution of participants by gender")
plt.xlabel("Gender")
plt.ylabel("Number of participants")
plt.tight_layout()
plt.show()

**Interpretation:** 
- The graph shows the **male/female distribution** in the sample. 
- In this type of study, the two sexes are generally relatively balanced, but it is not always perfectly 50/50. 
- For Tinder, it's important to know if the participant base is **symmetrical** or if there is a **gender imbalance** (which can impact matches).

In [ ]:
# 3.2. Age distribution

plt.figure(figsize=(6, 4))
sns.histplot(data=df_eda, x="age", bins=20, kde=True)
plt.title("Age distribution of participants")
plt.xlabel("Age")
plt.ylabel("Frequency")
plt.tight_layout()
plt.show()

**Interpretation:** 
- The distribution shows in which **age group** the participants are mainly located (often 20–35 years old). 
- This fits Tinder's historical target quite well: young adults.
- For marketing, knowing that the majority is in a certain bracket allows you to’**adapt the tone of communication** (visuals, messages, functionalities).

In [ ]:
# 3.3. Divided according to race/origine 

if "race" in df_eda.columns:
    plt.figure(figsize=(6, 4))
    sns.countplot(data=df_eda, x="race")
    plt.title("Divided by race / origine (codes)")
    plt.xlabel("Code race (see key data)")
    plt.ylabel("Number of participants")
    plt.tight_layout()
    plt.show()

**Interpretation:** 
- Each bar corresponds to a **race/origin code** (the exact meaning is described in the data key). 
- We generally observe some majority categories and others in the minority. 
- For Tinder, this reminds us that the user base can be **demographically heterogeneous**, which will encourage us to think about the diversity of the profiles offered.

In [ ]:
# 3.4. Overall match rate (2nd date)

global_match_rate = df_eda[target_var].mean()
print(f"Rate of {target_var} (approx. probability of a 2nd date) : {global_match_rate:.2%}")

**Interpretation:** 
- The overall rate of `match` indicates the **average probability that a speed date will lead to a second mutual date**. 
- This rate is often relatively low (for example 15–30%), which reflects the fact that the participants are selective. 
- For Tinder, this shows that even in a context where people meet in real life, **conversion to “real match” is far from automatic**

## 4. Analysis of attraction criteria (according to sex)

Key question: 
> “What are the most important characteristics in a partner, 
> and is it different between men and women?” We use the columns **declared importance** (e.g.: `attr1_1`, `sinc1_1`, etc.) which measure how important each criterion is in a partner before the evening.

In [ ]:
# 4.1. Average stated importance by gender

if importance_cols:
    importance_by_gender = (
        df_eda.dropna(subset=importance_cols + ["gender_label"])
        .groupby("gender_label")[importance_cols]
        .mean()
        .T  
    )

    display(importance_by_gender)

In [ ]:
# 4.2. Comparative barplot men vs women for importance ratings

if importance_cols:
    imp_melt = (
        df_eda.dropna(subset=importance_cols + ["gender_label"])
        .melt(id_vars="gender_label", value_vars=importance_cols,
              var_name="Criterion", value_name="Importance")
    )

    # Rename for more readability
    mapping_imp = {
        "attr1_1": "Attractiveness",
        "sinc1_1": "Sincerity",
        "intel1_1": "Intelligence",
        "fun1_1": "Fun",
        "amb1_1": "Ambition",
        "shar1_1": "Shared interests"
    }
    imp_melt["Criterion lisible"] = imp_melt["Criterion"].map(mapping_imp).fillna(imp_melt["Criterion"])

    plt.figure(figsize=(8, 5))
    sns.barplot(data=imp_melt, x="Criterion lisible", y="Importance", hue="gender_label", ci=None)
    plt.title("Average importance of criteria in a partner, by gender")
    plt.xlabel("Criterion")
    plt.ylabel("Average importance")
    plt.legend(title="Gender")
    plt.tight_layout()
    plt.show()


**Interpretation:** 
- The table and graph show the **average importance** of each criterion for men vs women. 
- In general, we observe that men tend to **overvalue attractiveness and fun**, while women often attach a little more importance to **sincerity and intelligence**. 
- For Tinder, this means that messages and profile screens can be **differentiated by gender**: for example, putting more emphasis on personality and reliability for male profiles seen by women, and more on visual aspect or fun for female profiles seen by men.


##5. Perceived attractiveness vs real impact on the 2nd date 

Idea: Compare **what people say is important** (declared importance) with **what actually influences the match**. We'll look: 
- the rate of `match` according to the **notes received** on each criterion (Attractiveness, Intelligence, Fun, Shared interests, etc.) ;
- by categories (low /medium /high).

In [ ]:
# 5.1. Utility function to calculate match rate by score category

def match_rate_by_score_cat(data, score_col, target=target_var, bins=(0, 5, 8, 10),
                            labels=("Low (≤5)", "Medium (6–8)", "High (≥9)")):
    """
    Split a score into 3 categories and calculate the match rate for each category.
    """
    subset = data[[score_col, target]].dropna()
    subset[score_col + "_cat"] = pd.cut(subset[score_col], bins=bins, labels=labels, include_lowest=True)
    rates = subset.groupby(score_col + "_cat")[target].mean().reset_index()
    rates.rename(columns={target: "match_rate"}, inplace=True)
    return rates

score_cols_to_check = [c for c in ["attr", "intel", "fun", "shar"] if c in rating_cols]

score_cols_to_check

In [ ]:
# 5.2. Visualiszation

for col in score_cols_to_check:
    rates = match_rate_by_score_cat(df_eda, col)
    plt.figure(figsize=(5, 4))
    sns.barplot(data=rates, x=col + "_cat", y="match_rate")
    plt.title(f"Taux de {target_var} selon le niveau de {col}")
    plt.xlabel(f"Niveau de {col}")
    plt.ylabel(f"Taux de {target_var}")
    plt.ylim(0, 1)
    plt.tight_layout()
    plt.show()
    display(rates)

**Interpretation:**

- For each criterion (attractiveness, intelligence, fun, shared interests), we see how the **match rate increases with the score**.
- In general, l’**attractiveness** has a strong effect: dates with a high attractiveness rating have a significantly higher **match rate**.
- Other criteria like **fun** and **shared interests** also play an important role, sometimes almost as strong as appearance.
- The message for Tinder: don't just focus on the profile picture, but also on the **personality elements** that reinforce the desire for a second date.

##6. Shared interests vs same origin/race 

Question: 
> “Are shared interests more important than a shared racial background?” 

We go: 
- create shared interest score categories (low /medium /high); 
- compare their impact on the rate of `match` with the effect of having the **same race /origin** (`same_race`).

In [ ]:
# 6.1. Choice of the received or perceived 'shared interests' column

if "shar_o" in df_eda.columns:
    shared_score_col = "shar_o"  # received rating
elif "shar" in df_eda.columns:
    shared_score_col = "shar"    # rating given by the participant on shared interests
else:
    shared_score_col = None

shared_score_col

In [ ]:
# 6.2. Match rate by same_race

if "same_race" in df_eda.columns:
    same_race_rates = (
        df_eda.dropna(subset=["same_race", target_var])
        .groupby("same_race")[target_var]
        .mean()
        .reset_index()
    )
    same_race_rates["same_race_label"] = same_race_rates["same_race"].map({0: "Different origin", 1: "Same origin"})

    plt.figure(figsize=(5, 4))
    sns.barplot(data=same_race_rates, x="same_race_label", y=target_var)
    plt.title(f"Taux de {target_var} depending on same origin or not")
    plt.xlabel("")
    plt.ylabel(f"Taux de {target_var}")
    plt.ylim(0, 1)
    plt.tight_layout()
    plt.show()

    display(same_race_rates)

In [ ]:
# 6.3. Match rate by level of shared interests

if shared_score_col is not None:
    shared_rates = match_rate_by_score_cat(df_eda, shared_score_col)
    plt.figure(figsize=(5, 4))
    sns.barplot(data=shared_rates, x=shared_score_col + "_cat", y="match_rate")
    plt.title(f"Rate of {target_var} according to the level of shared interests ({shared_score_col})")
    plt.xlabel("Level of shared interests")
    plt.ylabel(f"Rate of {target_var}")
    plt.ylim(0, 1)
    plt.tight_layout()
    plt.show()

    display(shared_rates)

**Interpretation:** 
- The first graph shows if being of **same origin** increases the match rate a little: often the effect exists, but remains **moderate**. 
- The second graph shows the impact of **shared interests**: the match rate generally increases **strongly** when participants feel a lot in common. 
- For Tinder, this suggests that **highlighting common interests** (hobbies, cultural tastes, lifestyle) is **more powerful** than simply filtering by origin. - Concretely: enrich profiles with interest tags and **prioritize matches with high common interests**.

## 7. Do people accurately predict their 'market value'?

## Question:

> “Can we predict whether someone will be appreciated based on their own self-evaluation?”

We compare:

- **self-ratings** (e.g., `attr1_2` = how the person rates themselves);
- **ratings received** from others (e.g., `attr_o`).

In [ ]:
# 7.1. Correlation auto-note vs received rating

pairs = []
for self_col, other_col in zip(self_cols, others_rating_cols):
    if self_col in df_eda.columns and other_col in df_eda.columns:
        subset = df_eda[[self_col, other_col]].dropna()
        if len(subset) > 0:
            corr = subset.corr().iloc[0, 1]
            pairs.append({"self": self_col, "other": other_col, "corr": corr})

corr_df = pd.DataFrame(pairs)
display(corr_df)

In [ ]:
# 7.2. Correlation barplot

if not corr_df.empty:
    # Make labels more readable
    label_map_self = {
        "attr1_2": "Self Attractiveness",
        "sinc1_2": "Self Sincerity",
        "intel1_2": "Self Intelligence",
        "fun1_2": "Self Fun",
        "amb1_2": "Self Ambition",
        "shar1_2": "Self Shared interests"
    }
    corr_df["self_label"] = corr_df["self"].map(label_map_self).fillna(corr_df["self"])

    plt.figure(figsize=(7, 4))
    sns.barplot(data=corr_df, x="self_label", y="corr")
    plt.title("Corrélation auto-évaluation vs received rating")
    plt.xlabel("Dimension")
    plt.ylabel("Correlation (Pearson)")
    plt.axhline(0, color="black", linewidth=1)
    plt.tight_layout()
    plt.show()

In [ ]:
# 7.3. Scatterplot for an example : auto evaluated attractiveness vs received rating

if "attr1_2" in df_eda.columns and "attr_o" in df_eda.columns:
    subset_attr = df_eda[["attr1_2", "attr_o"]].dropna()
    
    plt.figure(figsize=(5, 4))
    sns.scatterplot(data=subset_attr, x="attr1_2", y="attr_o", alpha=0.4)
    plt.title("Self Attractiveness vs Attractiveness received")
    plt.xlabel("Self-rated attractiveness (attr1_2)")
    plt.ylabel("Note received attractivité (attr_o)")
    plt.tight_layout()
    plt.show()

**Interpretation:**

- Correlations are often **positive but moderate**: people who consider themselves very attractive / fun / intelligent tend, on average, to receive higher ratings from others, but the relationship is not perfect.
- This suggests that most people have a **generally accurate** intuition of their “market value,” though with individual cases of **overestimation and underestimation**.
- For Tinder, this opens the door to **profile calibration assistance** features: for example, suggesting better photos or descriptions to users who significantly overestimate or underestimate themselves compared with the feedback they receive.

## 8. Date order effect on success

Question :

> “Is it better to go early or late in the evening?”

We use the dating order position variable (`position` ou `order`, according to the data key).

In [ ]:
# 8.1. Column choice

pos_col = None
for c in ["position", "order"]:
    if c in df_eda.columns:
        pos_col = c
        break

pos_col

In [ ]:
# 8.2. Position distribution

if pos_col is not None:
    plt.figure(figsize=(6, 4))
    sns.histplot(data=df_eda, x=pos_col, bins=20, kde=False)
    plt.title("Distribution of date position during the evening")
    plt.xlabel("Position during the evening")
    plt.ylabel("Number of dates")
    plt.tight_layout()
    plt.show()

In [ ]:
# 8.3. Match rate by position quartile

if pos_col is not None:
    subset_pos = df_eda[[pos_col, target_var]].dropna()
    subset_pos["position_quartile"] = pd.qcut(subset_pos[pos_col], 4,
                                              labels=["Very early", "Early", "Late", "Very late"])
    pos_rates = (
        subset_pos.groupby("position_quartile")[target_var]
        .mean()
        .reset_index()
        .rename(columns={target_var: "match_rate"})
    )

    plt.figure(figsize=(6, 4))
    sns.barplot(data=pos_rates, x="position_quartile", y="match_rate")
    plt.title(f"Rate of {target_var} depending on timing during the evening")
    plt.xlabel("Moment during the evening")
    plt.ylabel(f"Rate of {target_var}")
    plt.ylim(0, 1)
    plt.tight_layout()
    plt.show()

    display(pos_rates)


**Interpretation:**

- The distribution of positions shows how many dates take place at the beginning, middle, or end of the evening.
- The quartile chart helps identify whether there is a **fatigue effect** (fewer matches toward the end of the evening) or, conversely, a **warm-up effect** (participants become more generous after a few dates).
- In these data, some differences can be observed, but they generally remain **moderate**: there is no huge “jackpot” advantage to going first or last, although certain moments may be slightly more favorable.
- For Tinder, this may inspire strategies regarding the **order in which profiles are displayed** (for example, avoiding overwhelming the user immediately or at the very end of a session).

## 9. Final Summary & Business Recommendations for Tinder

### 9.1. Main Factors That Increase the Probability of a Second Date

According to the analysis:

1. **Perceived Attractiveness**  
   - Dates with a high attractiveness rating have a **significantly higher match rate**.  
   - Visual appearance remains a strong decision driver.

2. **Fun & Friendliness**  
   - High ratings on the *Fun* criterion are strongly associated with a better match rate.  
   - Participants clearly value **pleasant and lighthearted interactions**, not only “serious” qualities.

3. **Shared Interests**  
   - When participants feel they have many things in common, the match rate increases **substantially**.  
   - This effect is often stronger than simply having the **same background**.

4. **Differences Between Men and Women**  
   - Men often report placing more importance on **attractiveness** and **fun**.  
   - Women tend to value **sincerity** and **intelligence** more highly.  
   - These differences are real but **not absolute**: all criteria matter on both sides.

5. **Self-Perception vs Reality**  
   - Correlations between self-evaluation and ratings received are **positive but modest**.  
   - Many people assess themselves fairly accurately, while some **overestimate or underestimate** themselves.

---

### 9.2. Concrete Recommendations for Tinder

1. **Highlight Shared Interests in Matching**  
   - Use hobbies, music tastes, activities, etc. to **prioritize highly compatible matches**.  
   - For example: “You have 7 shared interests, including hiking and concerts.”

2. **Adapt Profile Presentation by Gender**  
   - For profiles shown to women: place more emphasis on **reliability, sincerity, and intelligence** (bio, prompts, badges).  
   - For profiles shown to men: also highlight **visual appeal**, while not neglecting the **fun** and social side.

3. **Improve the Visibility of Fun and Friendliness**  
   - Add profile fields and bio examples that showcase **sense of humor** and a sociable personality.  
   - Highlight activity photos rather than overly static portraits.

4. **Use Feedback to Help Users Calibrate Their Profile**  
   - Based on matches and interactions, provide **simple feedback**, such as:  
     “People who match with you especially appreciate your humor / your interests / your outdoor photos.”  
   - Offer **profile optimization suggestions** (photos, bio, interests to add).

5. **Rethink the Order of Profile Display**  
   - Test strategies where sessions begin with profiles that have **high match potential** (many shared interests + strong estimated attractiveness).  
   - Avoid sessions that are too long and create **decision fatigue**, which may reduce matches.

---

### 9.3. Conclusion

> “Our analysis shows that a **second date** does not depend only on attractiveness,  
> but also strongly on perceived **fun** and **shared interests**.  
> Tinder would therefore benefit from enriching its profiles and algorithm  
> to highlight these dimensions, differentiated by gender, rather than  
> focusing only on the profile picture.”

This notebook clearly explains:

- how the data were **cleaned and prepared**;
- how we analyzed the **attraction criteria** and their **real effects** on matching;
- and how to derive **operational recommendations** for Tinder.